# XOR, MLPs & Softmax

**Companion lesson:** https://ml-viz.vercel.app/courses/neural-networks/04-xor-and-mlp

From the perceptron learning rule, through XOR, to a trained two-layer MLP with softmax output.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (7, 5)
np.random.seed(0)

## The perceptron learning rule

The perceptron updates weights **only on misclassified examples**: $\mathbf{w} \leftarrow \mathbf{w} + \eta(y - \hat{y})\mathbf{x}$.
It converges in finite steps on linearly separable data. On XOR it loops forever.

In [ ]:
import math

# AND dataset (linearly separable)
and_X = [[0,0],[1,0],[0,1],[1,1]]
and_y = [0, 0, 0, 1]

def perceptron_train(X, y, lr=1.0, max_epochs=50):
    w = [0.0, 0.0]; b = 0.0
    for epoch in range(max_epochs):
        errors = 0
        for xi, yi in zip(X, y):
            z = sum(w[j]*xi[j] for j in range(len(w))) + b
            y_hat = 1 if z > 0 else 0
            if y_hat != yi:
                delta = lr * (yi - y_hat)
                w = [w[j] + delta*xi[j] for j in range(len(w))]
                b += delta
                errors += 1
        if errors == 0:
            print(f'Converged at epoch {epoch+1}: w={[round(wi,2) for wi in w]}, b={round(b,2)}')
            return w, b
    print('Did NOT converge (non-separable data)')
    return w, b

print('--- AND ---')
w_and, b_and = perceptron_train(and_X, and_y)

# Test AND
for xi, yi in zip(and_X, and_y):
    z = sum(w_and[j]*xi[j] for j in range(2)) + b_and
    y_hat = 1 if z > 0 else 0
    print(f'  x={xi} -> pred={y_hat}, label={yi}, correct={y_hat==yi}')

print()
print('--- XOR ---')
xor_X = [[0,0],[1,0],[0,1],[1,1]]
xor_y = [0, 1, 1, 0]
w_xor, b_xor = perceptron_train(xor_X, xor_y, max_epochs=20)

## Visualising why XOR is not linearly separable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, X, y, title in [
    (axes[0], and_X, and_y, 'AND — linearly separable'),
    (axes[1], xor_X, xor_y, 'XOR — not linearly separable')
]:
    for xi, yi in zip(X, y):
        color = '#6366f1' if yi == 1 else '#f59e0b'
        marker = 'o' if yi == 1 else 's'
        ax.scatter(xi[0], xi[1], c=color, marker=marker, s=200, zorder=5)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.set_title(title)

# Draw the AND decision boundary
xs = np.linspace(-0.5, 1.5, 100)
if w_and[1] != 0:
    axes[0].plot(xs, [(-w_and[0]*x - b_and) / w_and[1] for x in xs],
                 '--', color='#94a3b8', linewidth=2, label='decision boundary')
    axes[0].legend()

axes[1].text(0.5, -0.4, 'No line can separate ● from ■', ha='center',
             color='#f87171', fontsize=10)
plt.tight_layout(); plt.show()

## XOR solved with a two-layer MLP

The hand-crafted weights from the lesson:
$\mathbf{W}_1 = [[1,1],[1,1]]$, $\mathbf{b}_1 = [0,-1]$, $\mathbf{W}_2 = [[1,-2]]$, $b_2 = 0$.

After the hidden layer applies ReLU, inputs $(1,0)$ and $(0,1)$ map to the same
hidden representation $(1, 0)$ — the network has learned they are equivalent.

In [ ]:
def relu(x): return [max(0.0, xi) for xi in x]
def sigmoid(z): return 1.0 / (1.0 + math.exp(-z))

W1 = [[1.0, 1.0], [1.0, 1.0]]
b1 = [0.0, -1.0]
W2 = [[1.0, -2.0]]
b2 = [0.0]

def mlp_xor(x):
    # Hidden layer
    z1 = [sum(W1[i][j]*x[j] for j in range(2)) + b1[i] for i in range(2)]
    h  = relu(z1)
    # Output layer
    z2 = sum(W2[0][j]*h[j] for j in range(2)) + b2[0]
    return sigmoid(z2), h

print('Input  | z1           | h (ReLU) | z2  | p(y=1)')
print('-' * 60)
for x, y in zip(xor_X, xor_y):
    p, h = mlp_xor(x)
    z1 = [sum(W1[i][j]*x[j] for j in range(2)) + b1[i] for i in range(2)]
    z2 = sum(W2[0][j]*h[j] for j in range(2)) + b2[0]
    pred = round(p)
    print(f'{x}  | {[round(v,1) for v in z1]}  | {[round(v,1) for v in h]}      | {z2:4.1f} | {p:.3f}  {'✓' if pred==y else '✗'}')

assert all(round(mlp_xor(x)[0]) == y for x, y in zip(xor_X, xor_y)), \
    'MLP must solve XOR perfectly'
print('VERIFY: two-layer MLP solves XOR.')

## Softmax & cross-entropy

For multi-class classification, the output layer applies softmax to convert raw logits
$\mathbf{z}$ to probabilities. Training minimizes cross-entropy loss $\mathcal{L} = -\log \hat{p}_y$.

In [ ]:
def softmax(z):
    z = [zi - max(z) for zi in z]   # numerical stability
    e = [math.exp(zi) for zi in z]
    s = sum(e)
    return [ei / s for ei in e]

def cross_entropy(probs, y):
    return -math.log(max(probs[y], 1e-12))

# 3-class example from the lesson
logits = [2.0, 1.0, -1.0]
probs = softmax(logits)
print('Logits:     ', logits)
print('Softmax:    ', [round(p, 3) for p in probs])
print('Sum to 1:   ', round(sum(probs), 12))
print()
print('Cross-entropy loss for true class 0:', round(cross_entropy(probs, 0), 4))

# Gradient: dL/dz_k = p_k - 1[k=y]
y_true = 0
grads = [probs[k] - (1 if k == y_true else 0) for k in range(3)]
print('Gradients dL/dz:', [round(g, 3) for g in grads])
print('Sum of gradients:', round(sum(grads), 12), '(must be 0)')

assert all(p > 0 for p in probs), 'Softmax outputs must be positive'
assert abs(sum(probs) - 1.0) < 1e-12, 'Softmax outputs must sum to 1'
assert abs(sum(grads)) < 1e-12, 'Cross-entropy gradients must sum to 0'
print('VERIFY: softmax/cross-entropy verified.')

## Temperature scaling

Dividing logits by temperature $T$ before softmax controls confidence:
small $T$ sharpens predictions, large $T$ flattens them.

In [ ]:
logits = [2.0, 1.0, -1.0]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

for T in temperatures:
    scaled = [z / T for z in logits]
    p = softmax(scaled)
    print(f'T={T:4.1f}: p = {[round(pi, 3) for pi in p]}')

## Key takeaways

- The **perceptron** is a linear classifier — it cannot solve XOR (not linearly separable).
- A **hidden layer** learns a new representation; two layers can solve any problem in principle.
- **Universal approximation**: one hidden layer of sufficient width can approximate any function.
- **Softmax** maps logits to probabilities; **cross-entropy** gives clean gradient $\hat{p}_k - \mathbb{1}[k=y]$.

## ✏️ Your turn

### Exercise 1 — Perceptron learning rule

The perceptron update rule: $\mathbf{w} \leftarrow \mathbf{w} + \eta(y - \hat{y})\mathbf{x}$, $b \leftarrow b + \eta(y - \hat{y})$.
Implement one update step and verify: no change on correct predictions, correct sign when misclassified.

In [ ]:
import numpy as np

def perceptron_step(w, b, x, y, lr=1.0):
    """One perceptron update step.
    w: weight array, b: bias float, x: input array, y: true label (0 or 1), lr: learning rate.
    Returns (new_w, new_b)."""
    # TODO(you): compute prediction (1 if w·x+b > 0 else 0), then apply the update rule
    ...

In [ ]:
w0 = np.array([1.0, -1.0])
b0 = 0.0

# Correctly classified — no update
x_correct = np.array([2.0, 0.0])   # w·x+b = 2 > 0, pred=1, y=1
new_w, new_b = perceptron_step(w0, b0, x_correct, y=1)
assert np.allclose(new_w, w0) and new_b == b0, \
    "no update when prediction is correct"

# Misclassified: pred=0, y=1 → weights should increase in direction of x
x_miss = np.array([0.5, 2.0])   # w·x+b = 0.5 - 2 = -1.5 < 0, pred=0, y=1
new_w2, new_b2 = perceptron_step(w0, b0, x_miss, y=1, lr=1.0)
assert np.allclose(new_w2, w0 + x_miss), \
    "weights should increase by x when y=1 and pred=0"
assert new_b2 == b0 + 1.0, \
    "bias should increase by lr when y=1 and pred=0"

# Misclassified: pred=1, y=0 → weights should decrease
x_miss2 = np.array([3.0, 0.0])   # w·x+b = 3 > 0, pred=1, y=0
new_w3, new_b3 = perceptron_step(w0, b0, x_miss2, y=0, lr=1.0)
assert np.allclose(new_w3, w0 - x_miss2), \
    "weights should decrease by x when y=0 and pred=1"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def perceptron_step(w, b, x, y, lr=1.0):
    y_hat = 1 if np.dot(w, x) + b > 0 else 0
    delta = lr * (y - y_hat)
    return w + delta * x, b + delta
```

</details>

### Exercise 2 — Softmax

Implement softmax and verify the three key properties: positivity, sum-to-one, and temperature scaling.

In [ ]:
import numpy as np

def softmax_np(z):
    """Numerically stable softmax. z: 1-D array. Returns probability array."""
    # TODO(you): shift by max for stability, then exp and normalise
    ...

In [ ]:
z = np.array([2.0, 1.0, -1.0])
p = softmax_np(z)

assert np.all(p > 0), "all probabilities must be positive"
assert abs(p.sum() - 1.0) < 1e-12, "probabilities must sum to 1"
assert p.argmax() == 0, "highest logit should get highest probability"

# Temperature=0.1 should sharpen (highest prob > 0.99)
p_sharp = softmax_np(z / 0.1)
assert p_sharp[0] > 0.99, "low temperature should give near one-hot output"

# Temperature=10 should flatten (all probs close to uniform)
p_flat = softmax_np(z / 10.0)
assert p_flat.max() - p_flat.min() < 0.1, "high temperature should flatten probabilities"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def softmax_np(z):
    e = np.exp(z - z.max())
    return e / e.sum()
```

</details>

### Exercise 3 — Cross-entropy gradient

The gradient of cross-entropy loss w.r.t. logits is $\partial \mathcal{L}/\partial z_k = \hat{p}_k - \mathbb{1}[k=y]$.
Verify: gradients sum to zero, correct class gradient is negative, incorrect class gradients are positive.

In [ ]:
import numpy as np

def cross_entropy_grad(z, y):
    """Gradient of cross-entropy loss w.r.t. logits z.
    z: 1-D logit array, y: integer true class index.
    Returns gradient array same shape as z."""
    # TODO(you): compute softmax, subtract 1 from the true class element
    ...

In [ ]:
z = np.array([2.0, 1.0, -1.0])
y_true = 0
grad = cross_entropy_grad(z, y_true)

assert abs(grad.sum()) < 1e-12, \
    "gradients must sum to 0 (probabilities sum to 1)"
assert grad[y_true] < 0, \
    "gradient for the true class must be negative (push logit up)"
assert all(grad[k] > 0 for k in range(3) if k != y_true), \
    "gradients for incorrect classes must be positive (push logits down)"

# Spot-check: for p=(0.705, 0.259, 0.035), grad[0] ≈ 0.705 - 1 = -0.295
p = softmax_np(z)
assert abs(grad[y_true] - (p[y_true] - 1)) < 1e-12, \
    "grad[y] must equal p[y] - 1"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cross_entropy_grad(z, y):
    p = softmax_np(z)
    p[y] -= 1
    return p
```

</details>